# MoP + DivPO v2 — Cross-Persona Pair Selection

## Why v2?

DivPO v1 measured rarity of each candidate **against its same-persona siblings** (e.g., the 4 other contrarian outputs for the same prompt). The evaluation metric is cross-persona SBERT cosine — how different are the 4 persona outputs from each other.

These are different axes. v1 DivPO could "win" training reward by generating lexically unusual outputs within a single persona, but all four personas could converge on the same unusual territory — destroying cross-persona diversity. Empirically confirmed: MoP+DivPO v1 has SBERT-cos = 0.70 vs MoP+SFT = 0.485 (higher = less diverse, DivPO hurt diversity).

**v2 fix**: rarity is computed against the **full cross-persona candidate pool** (all 4 personas × 4 candidates = 16 candidates). A candidate that differs from other personas' outputs gets high rarity. This directly aligns training with evaluation.

**Additional fix**: quality score no longer includes prompt-response cosine. Counter-arguments must semantically oppose the claim, so measuring cosine(prompt, response) punishes good counter-arguments. v2 uses coherence + length only.

## Kaggle settings

| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 (preferred) or A100 |
| Internet | On |
| Secret | `HF_TOKEN` = HuggingFace write token |

## Pipeline

| Cell | Phase | Est. time (T4 x2) |
|---|---|---|
| 1–2 | Install + setup | 5 min |
| 3 | Generate cross-persona candidates + select pairs | 45–90 min |
| 4 | Train 4 DivPO v2 adapters | 60–90 min |
| 5 | Verify | 5 min |

---
## Cell 1 — Install dependencies

In [ ]:
import os
INTERACTIVE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Interactive'

!pip install -q --upgrade transformers peft trl accelerate datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao

if INTERACTIVE:
    print('\n>>> INTERACTIVE: restart kernel once, then continue from Cell 2. <<<')
else:
    print('Save Version mode — continuing.')

---
## Cell 2 — Setup

In [ ]:
import os
import sys
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

REPO_DIR = '/kaggle/working/mop-divpo-llm-counter-argument'
REPO_URL = 'https://github.com/DasonTio/mop-divpo-llm-counter-argument.git'
BRANCH = 'feat/eval-pipeline'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} fetch origin {BRANCH} --prune
!git -C {REPO_DIR} checkout -B {BRANCH} origin/{BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PYTHONPATH'] = os.path.join(REPO_DIR, 'src')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
n_gpu = torch.cuda.device_count()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GB')
if n_gpu == 0:
    raise RuntimeError('No GPU. Set Accelerator to T4 x2 in Kaggle settings.')

print('Branch:')
!git status --short --branch
print(f'HF token: {os.environ["HF_TOKEN"][:8]}...')
print('Ready.')

---
## Cell 3 — Prepare DivPO v2 datasets (cross-persona pair selection)

This generates candidates from all 4 SFT adapters, then selects (chosen, rejected) pairs where rarity is measured against the **full cross-persona pool**.

- `--quality-weight 0.5 --rarity-weight 0.5`: balanced weights (v1 was 0.4/0.6, too rarity-heavy)
- `--min-quality 0.40`: slightly tighter quality gate than v1 (0.35)
- Output: `data/processed/divpo_v2/{persona}.jsonl`
- Push: to `DasonTio/mop-divpo-divpo-v2-data` (separate from v1 data)

In [ ]:
!python scripts/prepare_divpo_datasets.py \
    --cross-persona \
    --from-hub \
    --quality-weight 0.5 \
    --rarity-weight 0.5 \
    --min-quality 0.40 \
    --candidate-count 4 \
    --limit 500 \
    --gen-batch-size 32 \
    --output-dir data/processed/divpo_v2 \
    --push

In [ ]:
# Sanity check — print first pair from each persona
import json
from pathlib import Path

for persona in ['contrarian', 'systems_thinker', 'cross_domain_analogist', 'minimalist']:
    path = Path(f'data/processed/divpo_v2/{persona}.jsonl')
    if not path.exists():
        print(f'[{persona}] MISSING — data generation may have failed')
        continue
    with open(path) as f:
        recs = [json.loads(l) for l in f if l.strip()]
    if not recs:
        print(f'[{persona}] EMPTY')
        continue
    r = recs[0]
    m = r['metadata']
    print(f'[{persona}] {len(recs)} pairs | chosen_rarity={m["chosen_rarity"]} chosen_quality={m["chosen_quality"]} cross_persona={m.get("cross_persona")}')
    print(f'  chosen:   {r["chosen"][:120]}...')
    print(f'  rejected: {r["rejected"][:120]}...')
    print()

---
## Cell 4 — Train DivPO v2 adapters

Loads v2 preference data from local `data/processed/divpo_v2/`. Pushes adapters to
`DasonTio/mop-divpo-coauthor/divpo_v2/{persona}/`.

The only difference from v1 training is:
- `--dataset-dir data/processed/divpo_v2` (local v2 data, not HF Hub v1 data)
- `--output-stage divpo_v2` (separate subfolder on HF Hub)

In [ ]:
# Write accelerate config for 2-GPU DDP.
accel_cfg = """compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
downcast_bf16: 'no'
gpu_ids: all
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
use_cpu: false
"""
with open('/tmp/accel_config_v2.yaml', 'w') as f:
    f.write(accel_cfg)

import subprocess
from huggingface_hub import list_repo_files

MODEL_REPO = 'DasonTio/mop-divpo-coauthor'
PERSONAS = ['contrarian', 'systems_thinker', 'cross_domain_analogist', 'minimalist']

hub_files = set(list_repo_files(MODEL_REPO, repo_type='model', token=os.environ['HF_TOKEN']))

for persona in PERSONAS:
    if f'divpo_v2/{persona}/adapter_config.json' in hub_files:
        print(f'=== Skipping {persona} — divpo_v2 adapter already on Hub ===')
        continue
    print(f'=== Training DivPO v2: {persona} ===')
    cmd = [
        'accelerate', 'launch',
        '--config_file', '/tmp/accel_config_v2.yaml',
        '--num_processes', '2',
        '--num_machines', '1',
        '--mixed_precision', 'fp16',
        '--dynamo_backend', 'no',
        'scripts/train_divpo.py',
        '--persona', persona,
        '--dataset-dir', 'data/processed/divpo_v2',
        '--output-stage', 'divpo_v2',
        '--batch-size', '1',
        '--grad-accum', '32',
        '--max-length', '384',
    ]
    subprocess.run(cmd, check=True)

---
## Cell 5 — Verify DivPO v2 adapter

In [ ]:
import os
import sys
sys.path.insert(0, 'src')

from mop_divpo.inference.generate import generate

result = generate(
    prompt='Remote work is strictly better for productivity.',
    personas=['contrarian', 'minimalist'],
    adapter_stage='divpo_v2',
    token=os.environ['HF_TOKEN'],
    n=1,
    temperature=0.9,
    max_new_tokens=150,
    as_counter_argument=True,
)

for persona, outputs in result.items():
    print(f'--- {persona} ---')
    print(outputs[0])
    print()

---
## After this notebook: run evaluation

The `mop_divpo_v2` method is already registered in `run_baseline_evaluation.py`. Run:

```bash
python scripts/run_baseline_evaluation.py \
    --methods base prompt_only single_lora mop_sft mop_divpo mop_divpo_v2 \
    --only-generate
```

Then on any machine with an Anthropic API key:

```bash
python scripts/run_baseline_evaluation.py \
    --skip-generation \
    --judge anthropic
```

**What to look for in the table:**
- `mop_divpo_v2` SBERT-cos should be lower than `mop_sft` (0.485 is the bar to beat)
- `mop_divpo_v2` self-BLEU should be lower than `mop_divpo` v1 (currently 0.053)
- Quality (LLM-judge) should not regress below `mop_sft`